In [ ]:
import math, time
from quasar_client import QuasarClient
import numpy as np


client = QuasarClient.from_environment()
client.wait_for_simulation_start()



R = "robot.1"
# TODO
#not die(optional)
#работать в нескольких ячейках
#не сойти с ума(опционально)(провалено)


In [ ]:
import base64, hashlib, json
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey, X25519PublicKey
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

b64 = base64.b64decode

def signed(key, sig, text): #врет ли подпись ECDSA P-256/SHA-256
    try:
        key.verify(b64(sig), text.encode(), ec.ECDSA(hashes.SHA256()))
        return True
    except Exception:
        return False

def newer(a, b): #проверка новизны а относительно б. надо
    return [int(x) for x in a.split(".")]>[int(x) for x in b.split(".")]

def updates():
    idn = client.get_update_identity()
    if not idn["updatesAvailable"]:
        return
    me = X25519PrivateKey.from_private_bytes(b64(idn["devicePrivateKey"]))
    my_pub = me.public_key().public_bytes(serialization.Encoding.Raw, serialization.PublicFormat.Raw)
    disp = serialization.load_pem_public_key(idn["dispatcherPublicKey"].encode())
    have = {f["module"]:f["version"] for f in idn["firmware"]}

    for _ in range(3):
        ses = client.open_update_session()
        sid, dkey = ses["sessionId"], b64(ses["dispatcherKey"])
        if signed(disp, ses["signature"], sid + "|" +dkey.hex() + "|" + my_pub.hex()):
            break
        client.reject_update(sid, "подпись сеанса не сходится")
        client.journal("error", "CybP_02", "Подменный сеанс отклонен: подпись не сходится", sessionId = sid)
    secret = me.exchange(X25519PublicKey.from_public_bytes(dkey))
    aes = AESGCM(HKDF(hashes.SHA256(), 32, sid.encode(), b"moonbase-update-channel-v1").derive(secret))
    m = client.get_update_manifest(sid)
    man = json.loads(aes.decrypt(b64(m["nonce"]), b64(m["ciphertext"]), (sid + "|manifest").encode()))

    for p in man["packages"]:
        f = client.fetch_update(sid, p["id"])
        try:
            img = aes.decrypt(b64(f["nonce"]), b64(f["ciphertext"]), (sid + "|" + p["id"]).encode())
        except Exception as err:
            print("ошибка расшифровки: ", repr(err))
            img = b""
        h = hashlib.sha256(img).hexdigest()
        text = "|".join([p["id"], p["module"], p["version"], p["target"], p["sha256"]])
        if not img:
            why = "образ не расшифровывается"
        elif h != p["sha256"]:
            why = "SHA-256 образа не совпадает с манифестом"
        elif not signed(disp, p["signature"], text):
            why = "подпись пакета не сходится"
        elif p["target"] != idn["hardware"]:
            why = "пакет для другой модели: " + p["target"]
        elif not newer(p["version"], have.get(p["module"], "0.0.0")):
            why = "версия не новее установленной сейчас"
        else:
            why = None
        print(p["id"], p["module"], p["version"], p["target"], why)
        if why:
            client.reject_update(sid, why, p["id"])
            client.journal("error", "CybP_02", "Пакет отклонен: " + why, packageId = p["id"])
        else:
            client.install_update(sid, p["id"], h)
            client.wait_for_firmware()
            have[p["module"]] = p["version"]
            client.journal("info", "CybP_02", "Установлен " + p["module"] + " " + p["version"], packageId = p["id"])
        print("прошивки:", have)

updates()

import threading, struct, binascii
from quasar_client import QuasarRunStoppedError, QuasarConnectionError

L, RW = QuasarClient.YAW_ARM, QuasarClient.WHEEL_RADIUS
TPR, RPM = QuasarClient.TICKS_PER_REVOLUTION, QuasarClient.RATED_RPM
ZONES = {"зона конвертации": (55, 56, 64, 65, 73, 74),
         "зона генерации": (57, 58, 66, 67, 75, 76),
         "зона конфигурации": (59, 60, 61, 68, 69, 70, 77, 78, 79)}
lock = threading.RLock()
outbox = []
cur = [0.0, 0.0]
bads = {"cmd": [0, set(), 0.0], "fb": [0, set(), 0.0]}
cmd_seq = [None]
fb = {"seq": None, "ticks": None, "miss": 0, "kmin": None}
nav = {"x": None, "y": None, "h": None, "off": None, "fail": False, "ok": None, "rep": 0.0, "dbg": 0.0,
       "vx": 0.0, "vy": 0.0, "ts": 0}
dbg = {"scale": [], "rej": []}
rad = {"known": set(), "cell": None, "logged": False, "prev": 0.0}

def say(level, eid, msg, **d):
    with lock:
        outbox.append((level, eid, msg, d))

def bad(kind, reason):
    with lock:
        bads[kind][0] += 1
        bads[kind][1].add(reason)

def crc_ok(b):
    return binascii.crc_hqx(b[:-2], 0xFFFF) == struct.unpack("<H", b[-2:])[0]
unk = []
KIND ={"move_to": 3, "rotate_to": 4, "drive": 5}#hitier, hitier...

def motion(f, cmd):#godbless
    unk.append((f.hex(), dict(cmd)))
    k = cmd["kind"]
    if len(f) != 24 or f[0] != 0xA5 or f[1] != KIND[k] or f[3] != 0 or not crc_ok(f):
        return "заголовок или CRC"
    if cmd_seq[0] is not None and not 0 < (f[2] - cmd_seq[0]) % 256 < 10:
        return "повтор старого кадра"
    if k == "move_to":
        x, y, _, v = struct.unpack("<2i2H", f[4:16])
        bad_ = abs(x - cmd["x"] * 1e6) > 2 or abs(y - cmd["y"] * 1e6) > 2 or abs(v - cmd["speed"] * 1000) > 1
    elif k == "rotate_to":
        r, w = struct.unpack("<2h", f[4:8])
        bad_ = abs(math.remainder(r / 1e4 - cmd["rotation"], 2 * math.pi)) > 2e-4 or abs(w - cmd["angularSpeed"] * 1000) > 1
    else:
        d, v = struct.unpack("<2i", f[4:12])
        bad_ = abs(d - cmd["distance"] * 1e6) > 2 or abs(v - cmd["speed"] * 1000) > 1
    if bad_:
        return "параметры кадра не совпадают с командой"
    cmd_seq[0] = f[2]
    return None

            
        

def check_cmd(f, cmd):
    if cmd.get("kind") in KIND:
        return motion(f, cmd)
    if len(f) != 16 or f[0] != 0xA5 or f[1] != 0x01 or f[3] != 0 or f[12:14] != b"\0\0" or not crc_ok(f):
        return "заголовок или CRC"
    if cmd_seq[0] is not None and not 0 < (f[2] - cmd_seq[0]) % 256 < 10:
        return "повтор старого кадра"
    w = struct.unpack("<4h", f[4:12])
    if max(abs(v) for v in w) > RPM * 10 + 5:
        return "скорость колеса выше номинальной"
    vx, vy, a = cmd["x"], cmd["y"], cmd["angular"]
    n = math.hypot(vx, vy)
    if n >0.3:
        vx, vy = vx * 0.3/ n, vy * 0.3 / n
    a = max(-1.4, min(1.4, a))
    k = 300 / math.pi/ RW
    e = [(vx - vy - L *a) * k, (vx + vy + L * a) * k, (vx +vy - L *a) *k, (vx - vy + L * a)*k]
    m = max(abs(y) for y in e)
    ok = max(abs(x-y) for x,y in zip(w, e)) <= 3+0.02*m
    if not ok and m > 20:
        s = sum(x*y for x,y in zip(w, e))/sum(y *y for y in e)
        ok = 0.3 <= s <= 1.05 and max(abs(x - s*y) for x,y in zip(w, e)) <= 3+ 0.02 *m
    if ok:
        dbg["scale"].append((f[2], list(w), [round(v) for v in e]))
    if not ok:
        return " скорости колес не соответствуют команде"
    cmd_seq[0] = f[2]
    return None
        
def motor_filter(frame, command):
    why = check_cmd(frame, command)
    if why:
        bad("cmd", why)
        dbg["rej"].append((why, frame.hex()))
    return why

def check_fb(raw, ts):
    b = bytes.fromhex(raw)
    if len(b) != 24 or b[0] != 0xA5 or b[1] != 0x02 or b[3] != 0 or not crc_ok(b):
        return "заголовок или CRC", None
    seq, ticks = b[2], struct.unpack("<4i", b[4:20])
    temp = struct.unpack("<h", b[20:22])[0]/10
    if not -40 <= temp <= 120:
        return "нереальная температура", None
    lag = (round(ts/50) - seq)% 256
    if fb["seq"] is not None:
        n = (seq - fb["seq"]) % 256
        if n ==0:
            return None, None
        if n>= 128 or lag > fb["kmin"] + 2:
            return "повтор старого кадра", None
        if max(abs(t-o) for t,o in zip(ticks, fb["ticks"]))> RPM/60*TPR*0.05*n*1.3+20:
            return "скачок энкодера", None
    return None, (seq, ticks, lag)

def update(r):
    with lock:
        s = r["sensors"]
        why, val = check_fb(s["motorFeedback"], s["timestampMs"])
        if why:
            bad("fb", why)
            fb["miss"] +=1
            if fb["miss"] > 10:
                fb["sec"] = None
        gyro = s["gyro"]
        if gyro is None:
            return
        if r.get("x") is None:
            return
        
        if nav["off"] is None:
            nav["x"], nav["y"], nav["off"] = r["x"], r["y"], r["rotation"] - gyro["yaw"]
            nav["h"] = r["rotation"]
        h = gyro["yaw"] + nav["off"]
        fresh = False# imma gna kms riiiiiight now
        if val:
            seq, ticks, lag = val
            if fb["seq"] is not None:
                n = (seq - fb["seq"]) % 256
                d = [(t - o) / TPR * 2 * math.pi for t, o in zip(ticks, fb["ticks"])]
                dx = RW / 4 * (d[0] + d[1] + d[2] + d[3])
                dy = RW / 4 * (-d[0] + d[1] + d[2] - d[3])
                hm = nav["h"] + math.remainder(h - nav["h"], 2 * math.pi) / 2
                wx, wy = dx * math.cos(hm) - dy * math.sin(hm), dx * math.sin(hm) + dy * math.cos(hm)
                nav["x"] += wx
                nav["y"] += wy
                nav["vx"], nav["vy"] = wx / (n * 0.05), wy / (n * 0.05)
            fb["kmin"] = lag if fb["kmin"] is None else min(fb["kmin"], lag)
            fresh = lag == fb["kmin"]
            fb["seq"], fb["ticks"], fb["miss"] = seq, ticks, 0
            nav["ts"] = s["timestampMs"] - (lag - fb["kmin"]) * 50
        nav["h"] = h
        if fresh:
            compare(r)
            
def compare(r):
    ex, ey, eh = nav["x"], nav["y"], nav["h"]
    diff = math.hypot(r["x"] - ex, r["y"] - ey)
    dh = math.remainder(r["rotation"] - eh, 2* math.pi)
    now = time.monotonic()
    if not nav["fail"]:
        if not (ey > 2.40 and ex< 2.67):
            nav["x"], nav["y"] = r["x"], r["y"]
            nav["off"] += 0.1*dh
        elif diff > 0.02 or abs(dh) > 0.05:
            nav["fail"], nav["ok"], nav["rep"] = True, None, now
            c = QuasarClient.cell_at(ex, ey)
            zone = next((z for z, cells in ZONES.items() if c in cells), "реакторный блок")
            outbox.append(("nav", "local", ex, ey))
            outbox.append(("warning", "CybP_04", "Сбой глобальной навигации: " + zone + ", клетка " + str(c)
                           + ". Перехожу на локальную навигацию", {"zone": zone, "cell": c, "diff_m": round(diff, 3)}))
        else: 
            nav["dbg"] = max(nav["dbg"], diff)
    else:
        if diff < 0.025 and abs(dh) < 0.05:
            nav["ok"] = nav["ok"] or now
            if now - nav["ok"] > 1.5:
                nav["fail"] = False
                outbox.append(("nav", "global"))
                outbox.append(("warning", "CybP_04", " Глобальная навигация восстановлена, возвращаюсь к ней", {}))
        else:
            nav["ok"] = None
        if nav["fail"] and now - nav["rep"] > 0.5:
            nav["rep"] = now
            outbox.append(("nav", "local", ex, ey))

def pose(r):
    update(r)
    with lock:
        if nav["fail"]:
            t = min(0.6, (r["sensors"]["timestampMs"] - nav["ts"]) / 1000)
            return nav["x"] + nav["vx"]*t, nav["y"] + nav["vy"]*t, nav["h"]
    return r["x"], r["y"], r["rotation"]

def radiation(x, y, dose):
    c = QuasarClient.cell_at(x, y)
    depth = min( x%0.4, 0.4 - x%0.4, y % 0.4, 0.4 - y% 0.4)
    if c != rad["cell"]:
        rad["cell"], rad["logged"] = c, False
    if dose is None:
        return
    if dose > 3 and depth > 0.13 and dose >= 0.8 * rad["prev"]:
        rad["known"].add(c)
    rad["prev"] = dose
    if c in rad["known"] and not rad["logged"]:
        rad["logged"] = True
        say("warning", "CybP_03", " Аномальная клетка" + str(c) + ", доза" + str(round(dose, 1))
            + " МкЗв/ч, покидаю её", cell = c, dose = round(dose, 1))

def flush():
    now = time.monotonic()
    with lock:
        items = outbox[:]
        outbox.clear()
        for kind, text in (("cmd", "Заблокированы поврежденые кадры команд"),
                           ("fb", "Отброшены недостоверные кадры обратной связи")):
            n, reasons, last = bads[kind]
            if n and now - last >= 1:
                items.append(("warning", "CybP_05", text + ": " + str(n) + " (" + ", ".join(sorted(reasons)) + ")",
                             {"count": n}))
                bads[kind] = [0, set(), now]
    for it in items:
        if it[0] == "nav" and it[1] == "local":
            client.report_navigation(R, "local", *cur)
        elif it[0] == "nav":
            client.report_navigation(R, "global")
        else:
            client.journal(it[0], it[1], it[2], **it[3])

def mb_loop():
    while client.simulation_running and not client.stop_requested:
        t0 = time.monotonic()
        try:
            s = client.get_state()
            if s["attempt"]["phase"] == "finished":
                break
            r = s["robots"][0]
            x, y, h = pose(r)
            cur[:] = [x, y]
            radiation(x, y, (r["sensors"]["dosimeter"] or {}).get("doseRate"))
            flush()
        except (QuasarRunStoppedError, QuasarConnectionError):
            break
        except Exception as err:
            print("МБ:", repr(err))
        time.sleep(max(0.0, 0.1 - (time.monotonic() - t0)))

client.motor_filter = motor_filter
client.pose_provider = pose
threading.Thread(target=mb_loop, daemon = True).start()

            


In [ ]:
def open_gates():
    client.open_gate("gate.1")
    client.wait_until(lambda s: next(g["angle"] for g in s["gates"] if g["id"]=="gate.1") > 1.5, timeout=5)

def move(x, y, v=0.15, eps=0.01):
    client.move_to(R, x, y, speed=v, tolerance=eps, timeout=30)

def rotate(deg, v=1.4, eps=0.01):
    client.rotate_to(R, -deg * math.pi/180, angular_speed=v, angle_tolerance=eps, timeout=15)

rotate(-90)
move(0.6, 1.4)
move(0.6, 2.2)
move(1, 2.2)
open_gates()
move(1, 2.6)
move(1.1, 2.78, v=0.2)
# пока реактор не выключен едет
while not (client.get_state()["reactor"]["state"]=="off"):
    client.drive(R, 0.003, speed=0.02)
move(1, 2.6)
move(1, 2.2)
# начало езды к катушке (задача 2)
move(3, 2.2)
move(2.92, 2.97)
rotate(0)
client.set_magnet(R, True)

move(2.985, 2.97)

move(2.96, 0.35)
client.set_magnet(R, False)

# Езда по задаче 5 до ячеек но без сбора главной
move(2.96, 0.6)
rotate(-90)
move(3, 1.4)

client.set_magnet(R, True)
move(2.188, 1.5)

move(2.188, 1.57)

move(2.19, 1.4)
rotate(90)
move(0.6, 1.4)
move(0.6, 0.6)
move(1, 0.5)
client.set_magnet(R, False)
move(0.6, 0.6)
move(0.6, 1.4)
move(1.39, 1.4)
rotate(-90)
client.set_magnet(R, True)
move(1.385, 1.52)

move(1.385, 1.57)

move(1.39, 1.4)
rotate(90)
move(0.6, 1.4)
move(0.6, 0.6)
move(0.8, 0.52)
client.set_magnet(R, False)
move(0.6, 0.58)

# Начало езды к консольке (задача 4)

move(0.6, 2.2)
move(1, 2.2)
open_gates()
move(1, 2.8)
rotate(-90)
move(0.4, 2.8)

def solve_console(acc):
    time = np.asarray(acc["time"], dtype=float)
    cur = np.asarray(acc["current"], dtype=float)
    plateau = float(np.median(cur[20]))
    sm = np.convolve(cur, np.ones(3)/3, mode="same")
    threshold = plateau * 0.97
    k = next(idx for idx in range(1, len(sm)) if sm[idx] < threshold)
    switch = 0.5 * (time[k-1] + time[k])
    return {"current_amperes":round(plateau, 3), "switch_seconds":round(float(switch),2)}

answers = [solve_console(a) for a in client.get_console_task()["accumulators"]]
print(answers)
result = client.configure_charging(answers)
print(result["accepted"], result["attemptsLeft"])

# Задача с чёртовыми стержнями, чтоб они сломались
def touch(x, y):
    try:
        move(x,y)
    except Exception as e:
        print("упор:", str(e)[:50])

rods = [r["targetOffset"] for r in client.get_rod_targets()["rods"]]
print(rods)
move(2.36, 2.8)
client.set_magnet(R, True)
touch(2.36, 3.042)


client.drive(R, ((-rods[2])+0.012), speed=0.1)
while not (client.get_state()["rods"][2]["state"] == "configured"):
    client.drive(R, -0.003, 0.03)

client.set_magnet(R, False)

move(2.18, 2.8)
client.set_magnet(R, True)
touch(2.18, 3.042)


client.drive(R, ((-rods[1])+0.012), speed=0.1)
while not (client.get_state()["rods"][1]["state"] == "configured"):
    client.drive(R, -0.003, 0.03)

client.set_magnet(R, False)

move(2, 2.8)
client.set_magnet(R, True)
touch(2, 3.042)


client.drive(R, ((-rods[0])+0.012), speed=0.1)
while not (client.get_state()["rods"][0]["state"] == "configured"):
    client.drive(R, -0.003, 0.03)

client.set_magnet(R, False)

# прода с топливом
move(1.02, 2.75)
rotate(90)
open_gates()
move(1, 2.2)
move(0.6, 2.2)
move(0.6, 0.6)

# активация аккумов
move(2.05, 0.6)
client.wait_until(lambda s: s["charging"]["percent"] >= 100, timeout=400)
for acc in ("accumulator.1", "accumulator.2", "accumulator.3"):
    client.activate_accumulator(acc)

# обратно к ячейке
move(0.61, 0.55)
client.set_magnet(R, True)
client.wait_until(lambda s: s["assembler"]["state"] == "ready", timeout=100, interval=1.0)
move(0.61, 0.495)
move(0.6, 0.75)
rotate(-90)
move(0.6, 2.2)
move(1, 2.2)
open_gates()
move(1, 2.75)
move(0.5, 3.12)
client.set_magnet(R, False)
# возврат на базу (урааааа)
move(1.02, 2.75)
open_gates()
move(1, 2.2)
move(0.6, 2.2)
move(0.6, 1.4)
move(0.2, 1.4)
